<a href="https://colab.research.google.com/github/JiHyeonKu/JeonGiPeu/blob/main/conv7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os

files = ["productData.txt", "productLog.txt", "customerData.txt", "customerLog.txt"]
for f in files:
    if not os.path.exists(f):
        with open(f, 'w', encoding='utf-8') as file:
            pass

In [ ]:
import re
import sys
import os
import calendar
from datetime import datetime
from collections import Counter

# ── Windows 터미널 UTF-8 강제 적용 (§2.1) ────────────────────
if sys.platform == "win32":
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
    sys.stdin.reconfigure(encoding="utf-8")

# ── 파일 경로 상수 ────────────────────────────────────────────
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

PRODUCT_DATA  = os.path.join(BASE_DIR, "productData.txt")
PRODUCT_LOG   = os.path.join(BASE_DIR, "productLog.txt")
CUSTOMER_DATA = os.path.join(BASE_DIR, "customerData.txt")
CUSTOMER_LOG  = os.path.join(BASE_DIR, "customerLog.txt")
FILES = [PRODUCT_DATA, PRODUCT_LOG, CUSTOMER_DATA, CUSTOMER_LOG]

# 프로그램 전역 상태
current_datetime: datetime | None = None   # 6.1.4 현재 일시

# ══════════════════════════════════════════════════════════════
#  SECTION 1 : 파일 I/O 헬퍼
# ══════════════════════════════════════════════════════════════
def read_file(path: str) -> list[str]:
    if not os.path.exists(path):
        print(f"{path} 파일이 존재하지 않습니다.")
        sys.exit(1)
    with open(path, "r", encoding="utf-8", newline="") as f:
        return [line.rstrip("\r\n") for line in f if line.strip()]

def write_file(path: str, lines: list[str]) -> None:
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))

def append_line(path: str, line: str) -> None:
    with open(path, "a+", encoding="utf-8") as f:
        f.seek(0, os.SEEK_END)
        if f.tell() > 0:
            f.seek(f.tell() - 1)
            if f.read(1) != "\n":
                f.write("\n")
        f.write(line + "\n")

def parse_record(line: str) -> list[str]:
    return [field.strip() for field in line.split("|")]

# ══════════════════════════════════════════════════════════════
#  SECTION 2 : 유효성 검사 (4절 데이터 요소)
# ══════════════════════════════════════════════════════════════
DATETIME_RE = re.compile(
    r"^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])"
    r"\s([01]\d|2[0-3]):[0-5]\d:[0-5]\d$"
)
CATEGORIES      = {"냉동식품", "냉장식품", "과자", "음료", "주류", "생활용품", "기타"}
PAYMENT_METHODS = {"현금", "카드", "포인트"}
EVENT_TYPES     = {"IN", "OUT", "DISPOSE"}
CUSTOMER_NAME_RE = re.compile(r"^([가-힣]{2,5}|[a-zA-Z]{3,20})$")
EVENT_ID_RE      = re.compile(r"^[1-9][0-9]*$")
PURCHASE_ID_RE   = re.compile(r"^[1-9][0-9]*$")
PRODUCT_ID_RE    = re.compile(r"^\d{4}$")
CUSTOMER_ID_RE   = re.compile(r"^\d{4}$")

def is_valid_datetime(s: str) -> bool:
    if not DATETIME_RE.match(s):
        return False
    try:
        datetime.strptime(s, "%Y-%m-%d %H:%M:%S")
        return True
    except ValueError:
        return False

def parse_dt(s: str) -> datetime:
    return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")

def is_valid_product_id(s: str) -> bool:
    return bool(PRODUCT_ID_RE.match(s))

def validate_event_time(pid_list, event_time):
    products = _load_products()
    if current_datetime is None:
        print("현재 일시가 설정되지 않았습니다.")
        return False
    if event_time > current_datetime:
        print("이벤트 시각은 현재보다 미래일 수 없습니다.")
        return False
    for pid in pid_list:
        if pid not in products:
            print("존재하지 않는 상품입니다.")
            return False
        in_date = parse_dt(products[pid]["in_date"])
        if event_time < in_date:
            print("이벤트 시각은 입고일보다 이전일 수 없습니다.")
            return False
    return True

def is_valid_customer_id(s: str) -> bool:
    return bool(CUSTOMER_ID_RE.match(s))

def is_valid_category(s: str) -> bool:
    return s in CATEGORIES

def is_valid_price(s: str) -> bool:
    return s.isdigit()

def is_valid_customer_name(s: str) -> bool:
    return bool(CUSTOMER_NAME_RE.match(s))

def is_valid_event_id(s: str) -> bool:
    return bool(EVENT_ID_RE.match(s))

# ══════════════════════════════════════════════════════════════
#  SECTION 3 : 재고 계산 (이벤트 로그 기반)
# ══════════════════════════════════════════════════════════════
def calc_stock(product_id: str, as_of: datetime | None = None) -> int:
    stock = 0
    for line in read_file(PRODUCT_LOG):
        fields = parse_record(line)
        if len(fields) != 5:
            continue
        event_id, ids_raw, etype, qty_s, evt_time_s = fields
        ids = [i.strip() for i in ids_raw.split(",")]
        if product_id not in ids:
            continue
        if as_of and parse_dt(evt_time_s) > as_of:
            continue
        if etype == "IN":
            stock += ids.count(product_id)
        elif etype in ("OUT", "DISPOSE"):
            stock -= ids.count(product_id)
    return stock

def get_all_stocks(as_of: datetime | None = None) -> dict[str, int]:
    product_ids = {parse_record(l)[0] for l in read_file(PRODUCT_DATA)}
    return {pid: calc_stock(pid, as_of) for pid in product_ids}

def next_available_4digit_id(used_ids: set[str]) -> str | None:
    for num in range(1, 10000):
        candidate = str(num).zfill(4)
        if candidate not in used_ids:
            return candidate
    return None

def next_event_id() -> str:
    max_id = 0
    for line in read_file(PRODUCT_LOG):
        fields = parse_record(line)
        if fields and EVENT_ID_RE.match(fields[0]):
            max_id = max(max_id, int(fields[0]))
    return str(max_id + 1)

def next_purchase_id() -> str:
    max_id = 0
    for line in read_file(CUSTOMER_LOG):
        fields = parse_record(line)
        if fields and PURCHASE_ID_RE.match(fields[0]):
            max_id = max(max_id, int(fields[0]))
    return str(max_id + 1)

def next_product_id() -> str | None:
    used_ids = set()
    for line in read_file(PRODUCT_DATA):
        fields = parse_record(line)
        if fields and PRODUCT_ID_RE.match(fields[0]):
            used_ids.add(fields[0])
    return next_available_4digit_id(used_ids)

def next_customer_id() -> str | None:
    used_ids = set()
    for line in read_file(CUSTOMER_DATA):
        fields = parse_record(line)
        if fields and CUSTOMER_ID_RE.match(fields[0]) and fields[0] != "0000":
            used_ids.add(fields[0])
    return next_available_4digit_id(used_ids)

def last_event_time() -> datetime | None:
    times = []
    for path in (PRODUCT_LOG, CUSTOMER_LOG):
        for line in read_file(path):
            fields = parse_record(line)
            if len(fields) >= 5 and is_valid_datetime(fields[-1]):
                times.append(parse_dt(fields[-1]))
    return max(times) if times else None

# ══════════════════════════════════════════════════════════════
#  SECTION 4 : 무결성 검사 (5.9절)
# ══════════════════════════════════════════════════════════════
def ids_key(ids: list[str]) -> tuple[str, ...]:
    return tuple(sorted(pid.strip() for pid in ids))

def check_integrity() -> None:
    print("무결성 검사 중… ", end="", flush=True)
    errors = []

    product_ids = set()
    products = {}

    # ── productData.txt 검사 ─────────────────────────────
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        ok = (
            len(f) == 6
            and is_valid_product_id(f[0])
            and len(f[1]) >= 1
            and "\\" not in f[1]  # 백슬래시 이스케이프 문자 필터링
            and "/" not in f[1]
            and "|" not in f[1]
            and is_valid_price(f[2])  # 가격 검사 주석 해제됨
            and is_valid_category(f[3])
            and is_valid_datetime(f[4])
            and is_valid_datetime(f[5])
            and parse_dt(f[4]) <= parse_dt(f[5])
        )
        if not ok:
            errors.append(f"<{PRODUCT_DATA}> => {line}")
        else:
            if f[0] in product_ids:
                errors.append(f"<{PRODUCT_DATA}> 중복 상품ID => {line}")
            product_ids.add(f[0])
            products[f[0]] = {
                "name": f[1],
                "price": f[2],
                "category": f[3],
                "in_date": f[4],
                "exp_date": f[5],
            }

    # productLog와 customerLog의 파일 간 비교를 위해 정상 레코드를 따로 저장
    product_events = []
    purchase_records = []

    event_ids = set()
    # ── productLog.txt 검사 ─────────────────────────────
    for line in read_file(PRODUCT_LOG):
        f = parse_record(line)
        ids = []

        ok = (
            len(f) == 5
            and is_valid_event_id(f[0])
            and f[2] in EVENT_TYPES
            and f[3].isdigit() and int(f[3]) >= 1
            and is_valid_datetime(f[4])
        )

        if ok:
            ids = [i.strip() for i in f[1].split(",")]

            # 상품ID가 모두 productData.txt에 존재해야 함
            if any(pid not in product_ids for pid in ids):
                ok = False

            # 수량 필드는 상품ID 개수와 일치해야 함
            if int(f[3]) != len(ids):
                ok = False

            # 이벤트 시각은 상품의 입고일보다 이전일 수 없음
            if ok:
                event_time = parse_dt(f[4])
                for pid in ids:
                    if event_time < parse_dt(products[pid]["in_date"]):
                        ok = False
                        break

        if not ok:
            errors.append(f"<{PRODUCT_LOG}> => {line}")
        elif f[0] in event_ids:
            errors.append(f"<{PRODUCT_LOG}> 중복 이벤트ID => {line}")
        else:
            event_ids.add(f[0])
            product_events.append({
                "event_id": f[0],
                "ids": ids,
                "etype": f[2],
                "qty": int(f[3]),
                "time": f[4],
                "line": line,
            })

    customer_ids = set()

    # ── customerData.txt 검사 ─────────────────────────────
    for line in read_file(CUSTOMER_DATA):
        f = parse_record(line)
        ok = (
            len(f) == 3
            and is_valid_customer_id(f[0])
            and f[0] != "0000"
            and is_valid_customer_name(f[1])
            and f[2].isdigit()
        )
        if not ok:
            errors.append(f"<{CUSTOMER_DATA}> => {line}")
        elif f[0] in customer_ids:
            errors.append(f"<{CUSTOMER_DATA}> 중복 고객ID => {line}")
        else:
            customer_ids.add(f[0])

    purchase_ids = set()

    # ── customerLog.txt 검사 ─────────────────────────────
    for line in read_file(CUSTOMER_LOG):
        f = parse_record(line)
        ids = []

        ok = (
            len(f) == 7
            and PURCHASE_ID_RE.match(f[0])
            and is_valid_customer_id(f[1])
            and (f[1] == "0000" or f[1] in customer_ids)
            and is_valid_price(f[3])  # 가격 검사 주석 해제됨
            and f[4] in PAYMENT_METHODS
            and f[5].isdigit() and int(f[5]) >= 1
            and is_valid_datetime(f[6])
        )

        if ok:
            ids = [i.strip() for i in f[2].split(",")]

            # 모든 상품ID는 productData.txt에 존재해야 함
            if any(pid not in product_ids for pid in ids):
                ok = False

            # 수량 필드는 상품ID 개수와 일치해야 함
            if int(f[5]) != len(ids):
                ok = False

        if not ok:
            errors.append(f"<{CUSTOMER_LOG}> => {line}")
        elif f[0] in purchase_ids:
            errors.append(f"<{CUSTOMER_LOG}> 중복 구매ID => {line}")
        else:
            purchase_ids.add(f[0])
            purchase_records.append({
                "purchase_id": f[0],
                "customer_id": f[1],
                "ids": ids,
                "price": int(f[3]),
                "payment": f[4],
                "qty": int(f[5]),
                "time": f[6],
                "line": line,
            })

    # ── productLog OUT 이벤트와 customerLog 구매 기록 대응 검사 ──
    out_counter = Counter()
    out_line_map = {}

    for ev in product_events:
        if ev["etype"] == "OUT":
            key = (ids_key(ev["ids"]), ev["qty"], ev["time"])
            out_counter[key] += 1
            out_line_map.setdefault(key, ev["line"])

    purchase_counter = Counter()
    purchase_line_map = {}

    for pr in purchase_records:
        key = (ids_key(pr["ids"]), pr["qty"], pr["time"])
        purchase_counter[key] += 1
        purchase_line_map.setdefault(key, pr["line"])

    for key, count in out_counter.items():
        if purchase_counter[key] < count:
            errors.append(f"<{PRODUCT_LOG}> OUT 이벤트에 대응되는 구매 기록 불일치 => {out_line_map[key]}")

    for key, count in purchase_counter.items():
        if out_counter[key] < count:
            errors.append(f"<{CUSTOMER_LOG}> 구매 기록에 대응되는 OUT 이벤트 불일치 => {purchase_line_map[key]}")

    # ── 실제 재고 흐름 검사 ─────────────────────────────
    # 이벤트 시각 순서대로 IN/OUT/DISPOSE를 반영했을 때 재고가 음수가 되면 오류
    stock_by_pid = {pid: 0 for pid in product_ids}

    sorted_events = sorted(
        product_events,
        key=lambda ev: (parse_dt(ev["time"]), int(ev["event_id"]))
    )

    for ev in sorted_events:
        id_counts = Counter(ev["ids"])

        if ev["etype"] == "IN":
            for pid, count in id_counts.items():
                stock_by_pid[pid] += count

        elif ev["etype"] in ("OUT", "DISPOSE"):
            flow_ok = True

            for pid, count in id_counts.items():
                if stock_by_pid.get(pid, 0) < count:
                    errors.append(f"<{PRODUCT_LOG}> 재고 흐름 불일치 => {ev['line']}")
                    flow_ok = False
                    break

            if flow_ok:
                for pid, count in id_counts.items():
                    stock_by_pid[pid] -= count

    if errors:
        print("데이터 오류가 발생했습니다.")
        for e in errors:
            print(e)
        sys.exit(1)
    else:
        print("완료")

# ══════════════════════════════════════════════════════════════
#  SECTION 5 : 초기 설정 (6.1절)
# ══════════════════════════════════════════════════════════════
def startup_banner() -> None:
    print("=" * 60)
    print("'스마트 무인 편의점' 재고 관리 프로그램 v1.0")
    print("=" * 60)

def check_files() -> None:
    for f in FILES:
        if not os.path.exists(f):
            print("필수 파일이 존재하지 않습니다.")
            sys.exit(1)
        if not os.access(f, os.R_OK | os.W_OK):
            print("파일 접근 권한이 없습니다.")
            sys.exit(1)

def input_current_datetime() -> datetime:
    last = last_event_time()
    while True:
        print()
        s = input("현재 일시를 입력하세요 (YYYY-MM-DD HH:MM:SS):\n> ").strip()
        print()
        if not is_valid_datetime(s):
            print("날짜 형식이 올바르지 않습니다. (예: 2026-04-02 14:00:00)")
            continue
        dt = parse_dt(s)
        if last and dt <= last:
            print("현재 일시는 기존 로그의 마지막 시각보다 늦어야 합니다.")
            continue
        print("기준 시각이 설정되었습니다.")
        return dt

def auto_dispose(now: datetime) -> None:
    expired = []
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        if len(f) == 6 and parse_dt(f[5]) < now:
            pid = f[0]
            stock = calc_stock(pid)
            if stock >= 1:
                expired.append((pid, stock))

    if not expired:
        print("현재 폐기 처리할 유통기한 경과 상품이 없습니다.")
        return

    eid = next_event_id()
    ids_list = []
    for pid, stock in expired:
      ids_list.extend([pid] * stock)

    ids_field = ", ".join(ids_list)
    total_qty = len(ids_list)
    time_s = now.strftime("%Y-%m-%d %H:%M:%S")
    record = f"{eid}|{ids_field}|DISPOSE|{total_qty}|{time_s}"
    try:
        append_line(PRODUCT_LOG, record)
    except OSError:
        print("데이터 파일 접근에 실패하여 폐기 처리를 중단합니다.")
        sys.exit(1)
    print(f"총 {total_qty}개의 재고가 유통기한 경과로 자동 폐기 처리되었습니다.")

# ══════════════════════════════════════════════════════════════
#  SECTION 6 : 메인 메뉴 (6.2절)
# ══════════════════════════════════════════════════════════════
def main_menu() -> None:
    global current_datetime
    while True:
        print("\n1. 고객 관리")
        print("2. 재고 관리")
        print("3. 판매 처리")
        print("4. 가상의 일시 기준 재고 조회")
        print("0. 종료")
        choice = input("메뉴 > ").strip()
        if not choice.isdigit():
            print("‘1’,’2’,’3’,’4’,’0’ 중 올바른 메뉴 번호를 입력해주세요.")
            continue

        action_performed = False
        match choice:
            case "1":
                action_performed = menu_customer()
            case "2":
                action_performed = menu_inventory()
            case "3":
                action_performed = menu_sales()
            case "4":
                menu_virtual_stock()
                action_performed = True
            case "0":
                print("프로그램을 종료합니다.")
                sys.exit(0)
            case _:
                print("‘1’,’2’,’3’,’4’,’0’ 중 올바른 메뉴 번호를 입력해주세요.")
                continue

        if action_performed:
            current_datetime = input_current_datetime()
            auto_dispose(current_datetime)

# ══════════════════════════════════════════════════════════════
#  SECTION 7 : 고객 관리 (6.3절)
# ══════════════════════════════════════════════════════════════
def menu_customer() -> bool:
    while True:
        print("\n1. 고객 등록\n2. 고객 조회\n0. 메인 메뉴")
        choice = input("메뉴 > ").strip()

        if not choice.isdigit():
            print("‘1’,’2’,’0’ 중 올바른 메뉴 번호를 입력해주세요.")
            continue

        match choice:
            case "1":
                register_customer()
                return True
            case "2":
                search_customer()
                return True
            case "0":
                return False
            case _:
                print("‘1’,’2’,’0’ 중 올바른 메뉴 번호를 입력해주세요.")

def register_customer() -> None:
    while True:
        name = input("고객명: ").strip()
        if not is_valid_customer_name(name):
            print("고객명은 한글 2~5자 또는 영문 3~20자여야 합니다.")
            continue
        break
    new_id = next_customer_id()
    if new_id is None:
        print("사용 가능한 고객ID가 없어 고객 등록을 진행할 수 없습니다.")
        return
    print(f"\n고객ID: {new_id}  고객명: {name}  초기 포인트: 0")

    while True:
        confirm = input("등록하시겠습니까? (Y/N): ").strip()
        if confirm in ("Y", "y"):
            append_line(CUSTOMER_DATA, f"{new_id}|{name}|0")
            print("고객 등록이 완료되었습니다.")
            return
        elif confirm in ("N", "n"):
            print("등록을 취소했습니다.")
            return
        else:
            print("Y 또는 N으로 입력해 주세요.")

def search_customer() -> None:
    while True:
        raw = input("검색 (예: 1 1023 또는 2 김철수): ").strip()

        if " " not in raw:
            print("형식에 맞게 입력해 주세요. (예: 1 1023 또는 2 김철수)")
            continue

        if raw.startswith(" ") or raw.endswith(" "):
            print("형식에 맞게 입력해 주세요. (예: 1 1023 또는 2 김철수)")
            continue

        parts = raw.split(" ", 1)

        if parts[0] not in ("1", "2") or parts[1].startswith(" "):
            print("형식에 맞게 입력해 주세요. (예: 1 1023 또는 2 김철수)")
            continue

        criterion, keyword = parts[0], parts[1].strip()
        is_valid = True

        if criterion == "1":
            if not is_valid_customer_id(keyword):
                print("고객ID는 4자리 숫자로 이루어진 문자열이어야 합니다.")
                is_valid = False
        elif criterion == "2":
            if not is_valid_customer_name(keyword):
                print("고객명은 한글 2~5자 또는 영문 3~20자여야 합니다.")
                is_valid = False

        if not is_valid:
            continue

        break

    results = []
    for line in read_file(CUSTOMER_DATA):
        f = parse_record(line)
        if len(f) != 3:
            continue
        if criterion == "1" and f[0] == keyword:
            results.append(f)
        elif criterion == "2" and f[1] == keyword:
            results.append(f)

    if not results:
        print("검색 결과가 없습니다.")
        return
    results.sort(key=lambda x: x[0])
    print(f"\n{'고객ID':<8} {'고객명':<12} {'포인트'}")
    print("-" * 32)
    for f in results:
        print(f"{f[0]:<8} {f[1]:<12} {f[2]}")
    print(f"\n총 {len(results)}건 검색됨")

# ══════════════════════════════════════════════════════════════
#  SECTION 8 : 재고 관리 (6.4절)
# ══════════════════════════════════════════════════════════════
def menu_inventory() -> bool:
    while True:
        print("\n1. 상품 입고\n2. 재고 조회\n3. 전체 재고 출력\n4. 폐기 상품 조회\n0. 메인 메뉴")
        choice = input("메뉴 > ").strip()
        if not choice.isdigit():
            print("‘1’,’2’,’3’,’4’,’0’ 중 올바른 메뉴 번호를 입력해주세요.")
            continue
        match choice:
            case "1":
                receive_product()
                return True
            case "2":
                search_inventory()
                return True
            case "3":
                print_all_inventory()
                return True
            case "4":
                search_disposed()
                return True
            case "0":
                return False
            case _:
                print("‘1’,’2’,’3’,’4’,’0’ 중 올바른 메뉴 번호를 입력해주세요.")

def receive_product() -> None:
    while True:
        name = input("상품명: ").strip()
        if len(name) < 1 or "\\" in name or "/" in name or "|" in name:
            print("상품명은 1글자 이상이어야 하며 '\\', '/', '|'를 포함할 수 없습니다.")
            continue
        break
    while True:
        cat = input(f"카테고리 {sorted(CATEGORIES)}: ").strip()
        if not is_valid_category(cat):
            print("정확한 카테고리 문자열 중 하나를 입력해 주세요.")
            continue
        break
    while True:
        price_s = input("상품 가격: ").strip()
        if not is_valid_price(price_s):
            print("상품 가격은 0 이상의 정수여야 합니다.")
            continue
        break
    while True:
        in_date = input("입고일 (YYYY-MM-DD HH:MM:SS): ").strip()
        if not is_valid_datetime(in_date):
            print("날짜 형식이 올바르지 않습니다. (예: 2026-03-25 14:00:00)")
            continue
        if parse_dt(in_date) > current_datetime:
            print("입고일은 현재 일시 이전이어야 합니다.")
            continue
        break
    while True:
        exp_date = input("유통기한 (YYYY-MM-DD HH:MM:SS): ").strip()
        if not is_valid_datetime(exp_date):
            print("날짜 형식이 올바르지 않습니다. (예: 2026-06-30 23:59:59)")
            continue
        if parse_dt(exp_date) <= parse_dt(in_date):
            print("유통기한은 입고일 이후여야 합니다.")
            continue
        break

    new_pid = next_product_id()
    if new_pid is None:
        print("사용 가능한 상품ID가 없어 상품 입고를 진행할 수 없습니다.")
        return
    qty = 1

    print(f"""
    상품ID: {new_pid}
    상품명: {name}
    카테고리: {cat}
    가격: {price_s}
    입고일: {in_date}
    유통기한: {exp_date}
    """)

    while True:
        confirm = input("입고하시겠습니까? (Y/N): ").strip()
        if confirm in ("Y", "y"):
            append_line(PRODUCT_DATA, f"{new_pid}|{name}|{price_s}|{cat}|{in_date}|{exp_date}")
            eid = next_event_id()
            time_s = current_datetime.strftime("%Y-%m-%d %H:%M:%S")
            ids = [new_pid] * qty
            ids_field = ", ".join(ids)

            if not validate_event_time(ids, current_datetime):
                return

            append_line(PRODUCT_LOG, f"{eid}|{ids_field}|IN|{qty}|{time_s}")
            print("상품 입고가 완료되었습니다.")
            return

        elif confirm in ("N", "n"):
            print("입고를 취소했습니다.")
            return
        else:
            print("Y 또는 N으로 입력해 주세요.")

def _load_products() -> dict:
    products = {}
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        if len(f) == 6:
            products[f[0]] = {
                "name": f[1], "price": int(f[2]),
                "category": f[3], "in_date": f[4], "exp_date": f[5]
            }
    return products

def search_inventory() -> None:
    criteria_map = {"1": "상품ID", "2": "상품명", "3": "상품 가격",
                    "4": "카테고리", "5": "입고일", "6": "유통기한"}
    while True:
        raw = input("검색 기준번호 검색어 (예: 2 콜라): ").strip()

        # 공백이 아예 없는 경우 예외 처리
        if " " not in raw:
            print("형식에 맞게 입력해 주세요. (예: 2 콜라)")
            continue

        parts = raw.split(" ", 1)

        # 기준 번호가 유효하지 않거나, 검색어(parts[1])가 공백으로 시작하는 경우(표준 공백 2개 이상) 예외 처리
        if parts[0] not in criteria_map or parts[1].startswith(" "):
            print("형식에 맞게 입력해 주세요. (예: 2 콜라)")
            continue

        crit, keyword = parts[0], parts[1].strip()
        is_valid = True

        if crit == "1":
            if not is_valid_product_id(keyword):
                print("상품ID는 4자리 숫자로 이루어진 문자열이어야 합니다.")
                is_valid = False
        elif crit == "2":
            if len(keyword) < 1 or "\\" in keyword or "/" in keyword or "|" in keyword:
                print("상품명은 1글자 이상이어야 하며 '\\', '/', '|'를 포함할 수 없습니다.")
                is_valid = False
        elif crit == "3":
            test_kw = keyword.replace("~", "").replace("이하", "").replace("이상", "").replace(" ", "")
            if not test_kw.isdigit() and test_kw != "":
                print("상품 가격은 0 이상의 정수여야 합니다.")
                is_valid = False
        elif crit == "4":
            if not is_valid_category(keyword):
                print("정확한 카테고리 문자열 중 하나를 입력해 주세요.")
                is_valid = False
        elif crit == "5":
            test_kw = keyword.replace("~", "").replace("-", "").replace(":", "").replace(" ", "")
            if not test_kw.isdigit() and test_kw != "":
                print("날짜 검색 형식이 올바르지 않습니다.")
                is_valid = False
        elif crit == "6":
            test_kw = keyword.replace("~", "").replace("-", "").replace(":", "").replace(" ", "")
            if not test_kw.isdigit() and test_kw != "":
                print("유통기한 검색 형식이 올바르지 않습니다.")
                is_valid = False

        if not is_valid:
            continue

        break

    products = _load_products()
    stocks = get_all_stocks()
    matched = []
    for pid, p in products.items():
        if stocks.get(pid, 0) < 1:
            continue
        match crit:
            case "1":   hit = pid == keyword
            case "2":   hit = keyword in p["name"]
            case "3":   hit = _price_filter(keyword, p["price"])
            case "4":   hit = p["category"] == keyword
            case "5":   hit = _date_range_filter(keyword, p["in_date"])
            case "6":   hit = _date_range_filter(keyword, p["exp_date"])
            case _:
                hit = False
        if hit:
            matched.append((pid, p, stocks[pid]))

    if not matched:
        print("검색 결과가 없습니다.")
        return

    matched.sort(key=lambda x: x[0])
    _print_inventory_table(matched)

def _price_filter(keyword: str, price: int) -> bool:
    keyword = keyword.replace(" ", "")
    if "~" in keyword:
        parts = keyword.split("~", 1)
        lo = parts[0]
        hi = parts[1]
        lo_val = int(lo) if lo.isdigit() else 0
        hi_val = int(hi) if hi.isdigit() else float('inf')
        return lo_val <= price <= hi_val
    if keyword.endswith("이하") and keyword[:-2].isdigit():
        return price <= int(keyword[:-2])
    if keyword.endswith("이상") and keyword[:-2].isdigit():
        return price >= int(keyword[:-2])
    if keyword.isdigit():
        return str(price) == keyword
    return False

def parse_date_for_search(d_str: str, is_end: bool) -> str:
    d_str = d_str.strip()
    if not d_str:
        return "9999-12-31 23:59:59" if is_end else "0000-00-00 00:00:00"
    if len(d_str) == 4:
        return f"{d_str}-12-31 23:59:59" if is_end else f"{d_str}-01-01 00:00:00"
    if len(d_str) == 7:
        if is_end:
            try:
                y, m = int(d_str[:4]), int(d_str[5:7])
                last_day = calendar.monthrange(y, m)[1]
                return f"{d_str}-{last_day} 23:59:59"
            except ValueError:
                return f"{d_str}-31 23:59:59"
        else:
            return f"{d_str}-01 00:00:00"
    if len(d_str) == 10:
        return f"{d_str} 23:59:59" if is_end else f"{d_str} 00:00:00"
    if len(d_str) == 13:
        return f"{d_str}:59:59" if is_end else f"{d_str}:00:00"
    if len(d_str) == 16:
        return f"{d_str}:59" if is_end else f"{d_str}:00"
    return d_str

def _date_range_filter(keyword: str, date_s: str) -> bool:
    if "~" in keyword:
        parts = keyword.split("~", 1)
        lo = parse_date_for_search(parts[0], False)
        hi = parse_date_for_search(parts[1], True)
        return lo <= date_s <= hi
    else:
        lo = parse_date_for_search(keyword, False)
        hi = parse_date_for_search(keyword, True)
        return lo <= date_s <= hi

def _print_inventory_table(items: list) -> None:
    print(f"\n{'상품ID':<8}{'상품명':<16}{'카테고리':<12}{'가격':>8}{'수량':>6}  {'입고일':<22}{'유통기한'}")
    print("-" * 90)
    for pid, p, qty in items:
        print(f"{pid:<8}{p['name']:<16}{p['category']:<12}{p['price']:>8}{qty:>6}"
              f"  {p['in_date']:<22}{p['exp_date']}")
    print(f"\n총 {len(items)}개 상품")

def print_all_inventory() -> None:
    products = _load_products()
    stocks = get_all_stocks()
    in_stock = [(pid, p, stocks.get(pid, 0)) for pid, p in products.items()
                if stocks.get(pid, 0) >= 1]
    if not in_stock:
        print("등록된 상품이 없습니다.")
        return
    in_stock.sort(key=lambda x: x[0])
    _print_inventory_table(in_stock)
    print(f"총 {len(in_stock)}종  총 수량 {sum(q for _, _, q in in_stock)}개")

def search_disposed() -> None:
    products = _load_products()
    disposed = []
    for line in read_file(PRODUCT_LOG):
        f = parse_record(line)
        if len(f) == 5 and f[2] == "DISPOSE":
            disposed.append(f)
    if not disposed:
        print("폐기 처리된 상품 내역이 존재하지 않습니다.")
        return
    disposed.sort(key=lambda x: x[4], reverse=True)
    print(f"\n{'이벤트ID':<12}{'상품명':<30}{'폐기수량':>8}  {'이벤트 시각'}")
    print("-" * 72)
    for f in disposed:
        names = ", ".join(products.get(pid.strip(), {}).get("name", "?")
                          for pid in f[1].split(","))
        print(f"{f[0]:<12}{names:<30}{f[3]:>8}  {f[4]}")

# ══════════════════════════════════════════════════════════════
#  SECTION 9 : 판매 처리 (6.5절)
# ══════════════════════════════════════════════════════════════
def menu_sales() -> bool:
    print("\n[판매 처리]")

    product_ids = []
    product_map = {}
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        if len(f) == 6:
            product_map[f[0]] = f

    while True:
        pid = input("상품ID 입력 (종료: 0): ").strip()

        if pid == "0":
            if not product_ids:
                print("판매할 상품을 한 개 이상 선택해야 합니다.")
                continue
            break

        if not is_valid_product_id(pid):
            print("상품ID는 4자리 숫자로 이루어진 문자열이어야 합니다.")
            continue

        if pid not in product_map:
            print("해당 ID의 상품이 존재하지 않습니다.")
            continue

        if pid in product_ids:
            print("같은 상품ID를 중복 입력할 수 없습니다.")
            continue

        stock = calc_stock(pid)
        if stock <= 0:
            print("해당 상품의 재고가 없습니다.")
            continue

        exp_date = parse_dt(product_map[pid][5])
        if exp_date < current_datetime:
            print("유통기한 지난 상품입니다.")
            continue

        product_ids.append(pid)
        p = product_map[pid]
        print(f"상품명: {p[1]}, 카테고리: {p[3]}, 상품 가격: {p[2]}, 현재 재고 수량: {stock}")

    quantity = len(product_ids)
    total_price = sum(int(product_map[pid][2]) for pid in product_ids)

    while True:
        customer_id = input("고객ID 입력 (비회원: 0): ").strip()

        customers = {}
        for line in read_file(CUSTOMER_DATA):
            f = parse_record(line)
            if len(f) == 3:
                customers[f[0]] = {"name": f[1], "points": int(f[2])}

        if customer_id == "0":
            customer_id = "0000"
            break

        if customer_id == "0000":
            print("0000은 비회원 거래 기록용 ID이므로 회원ID로 입력할 수 없습니다.")
            continue

        if not is_valid_customer_id(customer_id):
            print("고객ID는 4자리 숫자로 이루어진 문자열이어야 합니다.")
            continue

        if customer_id not in customers:
            print("등록되지 않은 고객ID입니다.")
            continue

        print(f"고객명: {customers[customer_id]['name']}, 보유 포인트: {customers[customer_id]['points']}")
        break

    while True:
        if customer_id == "0000":
            payment = input("결제 방식 선택 (현금/카드): ").strip()
            if payment not in ("현금", "카드"):
                print("현금, 카드 중 하나를 입력해주세요.")
                continue
        else:
            payment = input("결제 방식 선택 (현금/카드/포인트): ").strip()
            if payment not in ("현금", "카드", "포인트"):
                print("현금, 카드, 포인트 중 하나를 입력해주세요.")
                continue

        if payment == "포인트" and customer_id != "0000":
            if customers[customer_id]["points"] < total_price:
                print(f"보유 포인트: {customers[customer_id]['points']}, 결제 필요 금액: {total_price}")
                print("포인트가 부족합니다.")
                continue

        break

    print(f"총 결제 금액: {total_price}원")

    if customer_id != "0000":
        current_points = customers[customer_id]["points"]

        if payment == "포인트":
            used_points = total_price
            remain_points = current_points - used_points

            print(f"차감될 포인트: {used_points}")
            print(f"결제 후 잔여 포인트: {remain_points}")
        else:
            earned_points = int(total_price * 0.01)

            print(f"적립 예정 포인트: {earned_points}")

    while True:
        confirm = input("결제하시겠습니까? (Y/N): ").strip()
        if confirm in ("Y", "y", "N", "n"):
            break
        print("Y 또는 N으로 입력해 주세요.")

    if confirm in ("N", "n"):
        print("결제를 취소했습니다.")
        return True

    if not validate_event_time(product_ids, current_datetime):
        return True

    if customer_id != "0000":
        if payment == "포인트":
            customers[customer_id]["points"] -= used_points
        else:
            customers[customer_id]["points"] += earned_points

        updated_lines = []
        for line in read_file(CUSTOMER_DATA):
            f = parse_record(line)
            if len(f) == 3 and f[0] == customer_id:
                updated_lines.append(f"{f[0]}|{f[1]}|{customers[customer_id]['points']}")
            else:
                updated_lines.append(line)
        write_file(CUSTOMER_DATA, updated_lines)

    eid = next_event_id()
    ids_field = ", ".join(product_ids)
    time_s = current_datetime.strftime("%Y-%m-%d %H:%M:%S")

    append_line(PRODUCT_LOG, f"{eid}|{ids_field}|OUT|{quantity}|{time_s}")

    purchase_id = next_purchase_id()
    append_line(
        CUSTOMER_LOG,
        f"{purchase_id}|{customer_id}|{ids_field}|{total_price}|{payment}|{quantity}|{time_s}"
    )

    print("결제가 완료되었습니다.")

    if customer_id != "0000":
        print("[포인트 변동 내역]")

        if payment == "포인트":
            print(f"차감 포인트: {used_points}")
            print(f"잔여 포인트: {customers[customer_id]['points']}")
        else:
            print(f"적립 포인트: {earned_points}")
            print(f"잔여 포인트: {customers[customer_id]['points']}")

    return True

# ══════════════════════════════════════════════════════════════
#  SECTION 10 : 가상의 일시 기준 재고 조회 (6.6절)
# ══════════════════════════════════════════════════════════════
def menu_virtual_stock() -> None:
    while True:
        s = input("가상의 일시 입력 (YYYY-MM-DD HH:MM:SS): ").strip()
        if not is_valid_datetime(s):
            print("올바른 날짜 형식이 아닙니다.")
            continue
        virtual_dt = parse_dt(s)
        break
    products = _load_products()
    stocks   = get_all_stocks(as_of=virtual_dt)
    available = [
        (pid, p, stocks[pid])
        for pid, p in products.items()
        if stocks.get(pid, 0) >= 1 and parse_dt(p["exp_date"]) >= virtual_dt
    ]
    available.sort(key=lambda x: x[0])
    print("\n[판매 가능한 재고 목록]")
    print(f"{'상품ID':<8} {'상품명':<20} {'수량'}")
    print("-" * 36)
    for pid, p, qty in available:
        print(f"{pid:<8} {p['name']:<20} {qty}")
    print(f"\n총 상품 개수: {len(available)}개")

# ══════════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════════
def main():
    global current_datetime
    startup_banner()
    check_files()
    check_integrity()
    current_datetime = input_current_datetime()
    auto_dispose(current_datetime)
    main_menu()

if __name__ == "__main__":
    main()

'스마트 무인 편의점' 재고 관리 프로그램 v1.0
무결성 검사 중… 완료

현재 일시를 입력하세요 (YYYY-MM-DD HH:MM:SS):
> 2020-02-02 02:02:02

기준 시각이 설정되었습니다.
현재 폐기 처리할 유통기한 경과 상품이 없습니다.

1. 고객 관리
2. 재고 관리
3. 판매 처리
4. 가상의 일시 기준 재고 조회
0. 종료
메뉴 > 2

1. 상품 입고
2. 재고 조회
3. 전체 재고 출력
4. 폐기 상품 조회
0. 메인 메뉴
메뉴 > 1
상품명: 콜라
카테고리 ['과자', '기타', '냉동식품', '냉장식품', '생활용품', '음료', '주류']: 음료
상품 가격: 1000
입고일 (YYYY-MM-DD HH:MM:SS): 1000-10-10 10:10:10
유통기한 (YYYY-MM-DD HH:MM:SS): 3030-03-03 03:03:03

    상품ID: 0001
    상품명: 콜라
    카테고리: 음료
    가격: 1000
    입고일: 1000-10-10 10:10:10
    유통기한: 3030-03-03 03:03:03
    
입고하시겠습니까? (Y/N): y
상품 입고가 완료되었습니다.

현재 일시를 입력하세요 (YYYY-MM-DD HH:MM:SS):
> 2020-02-02 02:02:03

기준 시각이 설정되었습니다.
현재 폐기 처리할 유통기한 경과 상품이 없습니다.

1. 고객 관리
2. 재고 관리
3. 판매 처리
4. 가상의 일시 기준 재고 조회
0. 종료
메뉴 > 2

1. 상품 입고
2. 재고 조회
3. 전체 재고 출력
4. 폐기 상품 조회
0. 메인 메뉴
메뉴 > 2
검색 기준번호 검색어 (예: 2 콜라): 2  콜라
형식에 맞게 입력해 주세요. (예: 2 콜라)
검색 기준번호 검색어 (예: 2 콜라): 2  /콜라\
형식에 맞게 입력해 주세요. (예: 2 콜라)
검색 기준번호 검색어 (예: 2 콜라): 2 \콜라
상품명은 1글자 이상이어야 하며 '\', '/', '|'를 포함할 수 없습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
